In [1]:
! mamba install --yes -c bioconda pyrpipe star>2.7.7a sra-tools>2.10.9 stringtie>2.1.4 trim-galore>0.6.6 orfipy>0.0.3 salmon>1.4.0

NB Mac M1 errors with conda and libarchive: might need to `cp /opt/miniconda3/bin/../lib/libarchive.19.dylib /opt/miniconda3/bin/../lib/libarchive.13.dylib`

In [29]:
! mamba install --yes sra-tools --force-reinstall


Looking for: ['sra-tools']

warning  libmamba Cache file "/opt/miniconda3/pkgs/cache/31ce02e0.json" was modified by another program
warning  libmamba Cache file "/opt/miniconda3/pkgs/cache/09cdf8bf.json" was modified by another program
warning  libmamba Cache file "/opt/miniconda3/pkgs/cache/35adf087.json" was modified by another program
[+] 0.0s
[+] 0.1s
bioconda/osx-64 (check zst) ━━━━━━━━╸━━━━━━━━   0.0 B @  ??.?MB/s Checking  0.1s[+] 0.2s
bioconda/osx-64 (check zst) ━━━━━━━━╸━━━━━━━━   0.0 B @  ??.?MB/s Checking  0.2sbioconda/osx-64 (check zst)                         Checked  0.3s
warning  libmamba Cache file "/opt/miniconda3/pkgs/cache/2a957770.json" was modified by another program
[+] 0.0s
bioconda/noarch (check zst) ━╸━━━━━━━━━━━━━━━   0.0 B @  ??.?MB/s Checking  0.0sbioconda/noarch (check zst)                         Checked  0.1s
warning  libmamba Cache file "/opt/miniconda3/pkgs/cache/7fb2ce72.json" was modified by another program
[+] 0.0s
pkgs/main/osx-64 (check zst)      

In [31]:
# Check that fasterq-dump has actually been installed, 
# and that the fastq-dump version is >2.10
! which fasterq-dump
! fastq-dump --version

/opt/miniconda3/bin/fasterq-dump

"fastq-dump" version 3.0.0



In [5]:
import os
os.makedirs("demo", exist_ok=True)
os.chdir("demo")

In [6]:
! wget ftp://ftp.ensemblgenomes.org/pub/release-46/plants/fasta/arabidopsis_thaliana/dna/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz
! gunzip Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz
! wget ftp://ftp.ensemblgenomes.org/pub/release-46/plants/gtf/arabidopsis_thaliana/Arabidopsis_thaliana.TAIR10.46.gtf.gz
! gunzip Arabidopsis_thaliana.TAIR10.46.gtf.gz

--2024-03-08 09:23:15--  ftp://ftp.ensemblgenomes.org/pub/release-46/plants/fasta/arabidopsis_thaliana/dna/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz
           => ‘Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz’
Resolving ftp.ensemblgenomes.org (ftp.ensemblgenomes.org)... 193.62.193.161
Connecting to ftp.ensemblgenomes.org (ftp.ensemblgenomes.org)|193.62.193.161|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /pub/release-46/plants/fasta/arabidopsis_thaliana/dna ... done.
==> SIZE Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz ... 36462703
==> PASV ... done.    ==> RETR Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz ... done.
Length: 36462703 (35M) (unauthoritative)

Arabidopsis_thalian 100%[===================>]  34.77M  61.4MB/s    in 0.6s    

2024-03-08 09:23:16 (61.4 MB/s) - ‘Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz’ saved [36462703]

--2024-03-08 09:23:16--  ftp://ftp.ensemblgenomes.org/

In [30]:
from pyrpipe import sra,qc,mapping,assembly
#define some vaiables
run_id='SRR976159'
working_dir='example_output'
gen='Arabidopsis_thaliana.TAIR10.dna.toplevel.fa'
ann='Arabidopsis_thaliana.TAIR10.46.gtf'
star_index='star_index/athaliana'
#initialize objects
#creates a star object to use with threads
star=mapping.Star(index=star_index,genome=gen,threads=4)
#use trim_galore for trimming
trim_galore=qc.Trimgalore()
#Stringtie for assembly
stringtie=assembly.Stringtie(guide=ann)
#create SRA object; this will download fastq if doesnt exist
srr_object=sra.SRA(run_id,directory=working_dir)
#create a pipeline using the objects
srr_object.trim(trim_galore).align(star).assemble(stringtie)

#The assembled transcripts are in srr_object.gtf
print('Final result',srr_object.gtf)

Start:24-03-08 09:57:41
$ prefetch -O example_output/SRR976159 SRR976159
End:24-03-08 09:57:42
Time taken:0:00:01
Following error occured executing above command (return code=3):
STDOUT:

2024-03-08T09:57:42 prefetch.3.0.0 sys: encryption failed while validating token within cryptographic module - Verification issue 0x4008 for this certificate: (  cert. version     : 3  serial number     : 00  issuer name       : C=US, O=The Go Daddy Group, Inc., OU=Go Daddy Class 2 Certification Authority  subject name      : C=US, O=The Go Daddy Group, Inc., OU=Go Daddy Class 2 Certification Authority  issued  on        : 2004-06-29 17:06:20  expires on        : 2034-06-29 17:06:20  signed using      : RSA with SHA1  RSA key size      : 2048 bits  basic constraints : CA=true  )
2024-03-08T09:57:42 prefetch.3.0.0 sys: encryption failed while validating token within cryptographic module - Verification issue 0x8 for this certificate: (  cert. version     : 3  serial number     : 1B:E7:15  issuer name   

ValueError: Please check fastq files None None

In [ ]:
! gzip example_output/SRR976159/*.fastq